<a href="https://colab.research.google.com/github/u78k89e34s23h67/pharmaguard/blob/main/pharmaguard_week1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Drive connected successfully")

Mounted at /content/drive
Drive connected successfully


In [ ]:
import os

folders = [
    '/content/drive/MyDrive/pharmaguard/data/cdsco',
    '/content/drive/MyDrive/pharmaguard/data/images/genuine',
    '/content/drive/MyDrive/pharmaguard/data/images/fake',
    '/content/drive/MyDrive/pharmaguard/models',
    '/content/drive/MyDrive/pharmaguard/notebooks',
    '/content/drive/MyDrive/pharmaguard/results',
]

for f in folders:
    os.makedirs(f, exist_ok=True)
    print(f"Created: {f}")

print("\nAll project folders created successfully")

Created: /content/drive/MyDrive/pharmaguard/data/cdsco
Created: /content/drive/MyDrive/pharmaguard/data/images/genuine
Created: /content/drive/MyDrive/pharmaguard/data/images/fake
Created: /content/drive/MyDrive/pharmaguard/models
Created: /content/drive/MyDrive/pharmaguard/notebooks
Created: /content/drive/MyDrive/pharmaguard/results

All project folders created successfully


In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"ukeshsurk23cs5034","key":"rtGcGWpxwdsTaDqA"}'}

In [ ]:
import os, shutil

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print("Kaggle connected successfully")

Kaggle connected successfully


In [ ]:
import os
result = os.popen('kaggle datasets list').read()
print(result[:300])

ref                                                             title                                                     size  lastUpdated                 downlo


print("=" * 40)
print("PHARMAGUARD — DAY 1 COMPLETE")
print("=" * 40)
print("✓ Google Colab — GPU ready")
print("✓ Google Drive — connected")
print("✓ Project folders — created")
print("✓ Kaggle API — connected")
print("=" * 40)
print("Tomorrow Day 2 — 7 PM")
print("Task: Download CDSCO data")
print("=" * 40)

In [ ]:
!pip install kaggle --upgrade -q
print("Kaggle upgraded successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.5/111.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 25.5 MB/s eta 0:00:00
Kaggle upgraded successfully


# Day 2 — CDSCO Data Download
**Date:** July 2, 2026  
**Week:** 1 of 18  
**Goal:** Download real Indian government drug data from CDSCO

## What we do today:
- Download NSQ alert PDFs from cdsco.gov.in
- Download spurious drug alerts
- Download banned drugs list
- Parse PDFs into clean CSV format
- Save everything to Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Drive connected — Day 2 starting")

Mounted at /content/drive
Drive connected — Day 2 starting


### Step 1 — Install PDF Parser Library
We need PyMuPDF to read CDSCO PDF files

In [2]:
# Install PDF reading library
!pip install pymupdf -q
print("PyMuPDF installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 84.0 MB/s eta 0:00:00
PyMuPDF installed successfully


### Step 2 — Download CDSCO NSQ Alert Data
Downloading real Indian government drug alert data

In [3]:
import requests
import os

# Folder to save CDSCO data
save_folder = '/content/drive/MyDrive/pharmaguard/data/cdsco'

# CDSCO NSQ Alert PDFs — real government links
nsq_urls = {
    'NSQ_Jan2024': 'https://cdsco.gov.in/opencms/export/sites/CDSCO_WEB/Drugs/NSQ/2024/January2024.pdf',
    'NSQ_Feb2024': 'https://cdsco.gov.in/opencms/export/sites/CDSCO_WEB/Drugs/NSQ/2024/February2024.pdf',
    'NSQ_Mar2024': 'https://cdsco.gov.in/opencms/export/sites/CDSCO_WEB/Drugs/NSQ/2024/March2024.pdf',
    'NSQ_Apr2024': 'https://cdsco.gov.in/opencms/export/sites/CDSCO_WEB/Drugs/NSQ/2024/April2024.pdf',
}

# Download each PDF
for name, url in nsq_urls.items():
    try:
        response = requests.get(url, timeout=30)
        if response.status_code == 200:
            filepath = os.path.join(save_folder, f'{name}.pdf')
            with open(filepath, 'wb') as f:
                f.write(response.content)
            print(f"✅ Downloaded: {name}")
        else:
            print(f"❌ Failed: {name} — Status {response.status_code}")
    except Exception as e:
        print(f"❌ Error: {name} — {e}")

print("\nCDSCO download attempt complete")

❌ Error: NSQ_Jan2024 — HTTPSConnectionPool(host='cdsco.gov.in', port=443): Max retries exceeded with url: /opencms/export/sites/CDSCO_WEB/Drugs/NSQ/2024/January2024.pdf (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7b46d5411df0>: Failed to resolve 'cdsco.gov.in' ([Errno -3] Temporary failure in name resolution)"))
❌ Failed: NSQ_Feb2024 — Status 404
❌ Failed: NSQ_Mar2024 — Status 404
❌ Failed: NSQ_Apr2024 — Status 404

CDSCO download attempt complete


### Note — Plan B
CDSCO website blocks direct HTTP requests from Colab.
Solution: Manual download + Kaggle dataset as backup.
This is documented as a known limitation.

### Step 3 — Parse All 6 CDSCO PDFs into Master CSV

In [2]:
!pip install pymupdf -q
print("PyMuPDF installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 80.2 MB/s eta 0:00:00
PyMuPDF installed


In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Drive connected")

Mounted at /content/drive
Drive connected


In [5]:
import fitz
import os

# Load all PDFs and extract text
pdf_folder = '/content/drive/MyDrive/pharmaguard/data/cdsco'
pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith('.pdf')]

print(f"Found {len(pdf_files)} PDF files:")
for f in pdf_files:
    print(f"  → {f}")

Found 6 PDF files:
  → NSQ ALERT FOR THE MONTH OF DEC-2024.pdf
  → NSQ ALERT FOR THE MONTH OF JUly-2024.pdf
  → alert August-2024.pdf
  → NSQ Alert For the month of  Sept-2024.pdf
  → Drug NSQ Alert April 2024.pdf
  → STATE NSQ ALERT FOR THE MONTH OF DEC-2024. (2).pdf


### Step 4 — Extract Text From All PDFs

In [6]:
import fitz
import os

pdf_folder = '/content/drive/MyDrive/pharmaguard/data/cdsco'
pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith('.pdf')]

# Extract text from all PDFs
all_text = {}

for pdf_file in pdf_files:
    pdf_path = os.path.join(pdf_folder, pdf_file)
    doc = fitz.open(pdf_path)

    text = ""
    for page in doc:
        text += page.get_text()

    all_text[pdf_file] = text
    print(f"✅ Extracted: {pdf_file} — {len(doc)} pages — {len(text)} characters")

print(f"\nTotal PDFs extracted: {len(all_text)}")

✅ Extracted: NSQ ALERT FOR THE MONTH OF DEC-2024.pdf — 9 pages — 14479 characters
✅ Extracted: NSQ ALERT FOR THE MONTH OF JUly-2024.pdf — 11 pages — 14584 characters
✅ Extracted: alert August-2024.pdf — 8 pages — 12539 characters
✅ Extracted: NSQ Alert For the month of  Sept-2024.pdf — 9 pages — 13163 characters
✅ Extracted: Drug NSQ Alert April 2024.pdf — 11 pages — 14950 characters
✅ Extracted: STATE NSQ ALERT FOR THE MONTH OF DEC-2024. (2).pdf — 18 pages — 30174 characters

Total PDFs extracted: 6


### Step 5 — Parse Drug Records Into Clean CSV

In [7]:
import pandas as pd
import re

all_records = []

for pdf_name, text in all_text.items():
    # Extract month/year from filename
    month = pdf_name.replace('.pdf', '')

    # Split into lines
    lines = text.split('\n')
    lines = [l.strip() for l in lines if l.strip()]

    i = 0
    while i < len(lines):
        line = lines[i]

        # Look for batch number pattern
        # Batch numbers are alphanumeric codes
        batch_match = re.search(r'\b([A-Z0-9]{4,15})\b', line)

        # Look for date pattern MM/YYYY
        date_match = re.search(r'\d{2}/\d{4}', line)

        if batch_match and date_match:
            # Try to get drug name from previous line
            drug_name = lines[i-1] if i > 0 else "Unknown"
            batch_no = batch_match.group(1)

            # Get manufacturer from next lines
            manufacturer = lines[i+1] if i+1 < len(lines) else "Unknown"

            all_records.append({
                'drug_name': drug_name,
                'batch_no': batch_no,
                'date': date_match.group(0),
                'manufacturer': manufacturer,
                'source_pdf': month,
                'status': 'NSQ/FAKE'
            })
        i += 1

# Create dataframe
df = pd.DataFrame(all_records)
df = df.drop_duplicates(subset=['batch_no'])

print(f"Total records extracted: {len(df)}")
print("\nFirst 5 records:")
print(df.head())

Total records extracted: 23

First 5 records:
    drug_name batch_no     date           manufacturer  \
0     DP24029     2024  04/2024                03/2026   
1     04/2024     2026  03/2026  M/s. Bengal Chemicals   
5     01/2024     2025  12/2025         M/s. Karnataka   
6   AGI-23216     2023  06/2023                05/2025   
19    09/2024     2027  08/2027     M/s. Ciron Drugs &   

                             source_pdf    status  
0   NSQ ALERT FOR THE MONTH OF DEC-2024  NSQ/FAKE  
1   NSQ ALERT FOR THE MONTH OF DEC-2024  NSQ/FAKE  
5   NSQ ALERT FOR THE MONTH OF DEC-2024  NSQ/FAKE  
6   NSQ ALERT FOR THE MONTH OF DEC-2024  NSQ/FAKE  
19  NSQ ALERT FOR THE MONTH OF DEC-2024  NSQ/FAKE  


### Step 6 — Improved PDF Parser

In [8]:
import pandas as pd
import re

all_records = []

for pdf_name, text in all_text.items():
    month = pdf_name.replace('.pdf', '')
    lines = text.split('\n')
    lines = [l.strip() for l in lines if l.strip()]

    i = 0
    while i < len(lines):
        line = lines[i]

        # Batch numbers in CDSCO follow specific patterns
        # Examples: DP24029, A4ITX004, 17800124, AGI-23216
        batch_pattern = re.search(
            r'\b([A-Z]{1,4}\d{4,8}|\d{6,10}|[A-Z]{2,4}-\d{4,6})\b',
            line
        )

        # Date pattern MM/YYYY
        date_pattern = re.search(r'(\d{2}/\d{4})', line)

        if batch_pattern and date_pattern:
            batch_no = batch_pattern.group(1)
            date_found = date_pattern.group(1)

            # Drug name — look back up to 3 lines
            drug_name = "Unknown"
            for back in range(1, 4):
                if i - back >= 0:
                    prev = lines[i - back]
                    # Drug names contain letters and spaces
                    if len(prev) > 5 and not re.search(r'\d{2}/\d{4}', prev):
                        drug_name = prev
                        break

            # Manufacturer — look forward
            manufacturer = "Unknown"
            for fwd in range(1, 4):
                if i + fwd < len(lines):
                    nxt = lines[i + fwd]
                    if 'M/s' in nxt or 'Ltd' in nxt or 'Pvt' in nxt:
                        manufacturer = nxt
                        break

            all_records.append({
                'drug_name'   : drug_name,
                'batch_no'    : batch_no,
                'date'        : date_found,
                'manufacturer': manufacturer,
                'source_pdf'  : month,
                'status'      : 'NSQ/FAKE'
            })
        i += 1

# Clean up
df = pd.DataFrame(all_records)
df = df.drop_duplicates(subset=['batch_no'])
df = df[df['drug_name'] != 'Unknown']
df = df.reset_index(drop=True)

print(f"Total clean records: {len(df)}")
print("\nSample records:")
print(df[['drug_name','batch_no','manufacturer','status']].head(10).to_string())

Total clean records: 8

Sample records:
           drug_name   batch_no             manufacturer    status
0           TABLETS)     470823  M/s.Uniplus Biotech (P)  NSQ/FAKE
1  Axetil Tablets IP  BT-227375    Pvt Ltd Plot No.B-29,  NSQ/FAKE
2    Tablets IP 40mg   PNTC3007                  Unknown  NSQ/FAKE
3      (20mg/75mg/75   MC240184   Pvt Ltd., Plot No. 79,  NSQ/FAKE
4     Injection USP-   PGY23118                  Unknown  NSQ/FAKE
5       Surgi Spirit  SS-084032                  Unknown  NSQ/FAKE
6          Cipladine   CHO40018                  Unknown  NSQ/FAKE
7           Ointment   CHO40020                  Unknown  NSQ/FAKE


### Step 7 — Better Parser Using Table Detection

In [9]:
import fitz
import pandas as pd
import re

all_records = []

pdf_folder = '/content/drive/MyDrive/pharmaguard/data/cdsco'
pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith('.pdf')]

for pdf_file in pdf_files:
    pdf_path = os.path.join(pdf_folder, pdf_file)
    doc = fitz.open(pdf_path)
    month = pdf_file.replace('.pdf','')

    for page_num in range(len(doc)):
        page = doc[page_num]

        # Extract words with their positions
        words = page.get_text("words")

        # Group words by their Y position (same row)
        rows = {}
        for word in words:
            x0, y0, x1, y1, text, block, line, wnum = word
            # Round Y to group words on same line
            y_key = round(y0 / 5) * 5
            if y_key not in rows:
                rows[y_key] = []
            rows[y_key].append((x0, text))

        # Sort rows by Y position
        for y_key in sorted(rows.keys()):
            row_words = sorted(rows[y_key], key=lambda x: x[0])
            row_text = ' '.join([w[1] for w in row_words])

            # Check if row contains batch number pattern
            batch = re.search(
                r'\b([A-Z]{0,4}\d{5,10}|[A-Z]{1,3}-\d{4,6}|[A-Z]{2,5}\d{3,8})\b',
                row_text
            )
            # Check if row contains date
            date = re.search(r'(\d{2}/\d{4})', row_text)

            if batch and date:
                all_records.append({
                    'raw_row'   : row_text,
                    'batch_no'  : batch.group(1),
                    'date'      : date.group(1),
                    'source'    : month,
                    'status'    : 'NSQ/FAKE'
                })

df_raw = pd.DataFrame(all_records)
df_raw = df_raw.drop_duplicates(subset=['batch_no'])
df_raw = df_raw.reset_index(drop=True)

print(f"Total records found: {len(df_raw)}")
print("\nFirst 10 raw rows:")
print(df_raw[['batch_no','date','raw_row']].head(10).to_string())

Total records found: 145

First 10 raw rows:
     batch_no     date                                                                                               raw_row
0     DP24029  04/2024          1. Cefotaxime DP24029 04/2024 03/2026 M/s. Bengal Chemicals Test of Clarity of Central Drugs
1    17800124  01/2024                         3. AMIKACIN 17800124 01/2024 12/2025 M/s. Karnataka Particulate Central Drugs
2   AGI-23216  06/2023  4. Rabeprazole AGI-23216 06/2023 05/2025 M/s. Alliaance Biotech, Sterility, Clarity of Central Drugs
3   AGI-23217  06/2023         5. Rabeprazole AGI-23217 06/2023 05/2025 M/s. Alliaance Biotech, pH, Clarity of Central Drugs
4  CT24250327  05/2024                  7. Fexofenadine CT24250327 05/2024 04/2026 M/s.CMG BIOTECH Dissolution Central Drugs
5    BCV22315  12/2023               12. BUPIVACAINE BCV22315 12/2023 11/2025 M/s. Nandani Medical Particulate Central Drugs
6      A21410  05/2024                            13. AXBEX A21410 05/2024 10/20

In [ ]:
import fitz
import os

pdf_folder = '/content/drive/MyDrive/pharmaguard/data/cdsco'
pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith('.pdf')]

# Count serial numbers in each PDF
# CDSCO tables use 1. 2. 3. format
# So highest serial number = actual record count

total_real = 0

for pdf_file in pdf_files:
    pdf_path = os.path.join(pdf_folder, pdf_file)
    doc = fitz.open(pdf_path)

    full_text = ""
    for page in doc:
        full_text += page.get_text()

    # Find all serial numbers like 1. 2. 3. etc
    serial_numbers = re.findall(r'^\s*(\d+)\.\s+\w', full_text, re.MULTILINE)

    if serial_numbers:
        max_serial = max([int(n) for n in serial_numbers])
    else:
        max_serial = 0

    print(f"{pdf_file[:40]}")
    print(f"  → Real record count: {max_serial}")
    total_real += max_serial

print(f"\nTotal real records across all PDFs: {total_real}")
print(f"Parser found: 145")
print(f"Difference: {total_real - 145} records missed")

In [11]:
import fitz
import os
import re

pdf_folder = '/content/drive/MyDrive/pharmaguard/data/cdsco'
pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith('.pdf')]

total_real = 0

for pdf_file in pdf_files:
    pdf_path = os.path.join(pdf_folder, pdf_file)
    doc = fitz.open(pdf_path)

    full_text = ""
    for page in doc:
        full_text += page.get_text()

    # Serial numbers in CDSCO are small numbers 1-200
    # Pattern: number followed by dot at START of line
    # And number must be less than 500
    serial_numbers = re.findall(r'^\s{0,5}(\d{1,3})\.\s+[A-Z]',
                                full_text, re.MULTILINE)

    # Filter only reasonable serial numbers
    valid_serials = [int(n) for n in serial_numbers if int(n) < 300]

    max_serial = max(valid_serials) if valid_serials else 0
    total_real += max_serial

    print(f"{pdf_file[:45]}")
    print(f"  → Real record count: {max_serial}")

print(f"\nTotal real records: {total_real}")
print(f"Parser found:       145")
print(f"Difference:         {total_real - 145}")

NSQ ALERT FOR THE MONTH OF DEC-2024.pdf
  → Real record count: 51
NSQ ALERT FOR THE MONTH OF JUly-2024.pdf
  → Real record count: 57
alert August-2024.pdf
  → Real record count: 137
NSQ Alert For the month of  Sept-2024.pdf
  → Real record count: 49
Drug NSQ Alert April 2024.pdf
  → Real record count: 3
STATE NSQ ALERT FOR THE MONTH OF DEC-2024. (2
  → Real record count: 84

Total real records: 381
Parser found:       145
Difference:         236
